In [33]:
import sys
import os
import requests
import pandas as pd
from io import BytesIO

from datetime import datetime, time
import altair as alt

%matplotlib inline
import matplotlib.pyplot as plt

sys.path.append(os.path.join(os.getcwd(), "..", "data"))

live_urls = [ "https://1drv.ms/x/c/d6aca2526f83594b/IQAMcIopdLU9SINiVRsgBHwWAZsXQJ5-PHSv1BevGvZ8f0Q?download=1",
              "https://1drv.ms/x/c/d6aca2526f83594b/IQAlE6FfCDS_QL0HXfdxWdtCAae3Ilx-a_K9hA_PXGN3Ofs?download=1"
               ]

urls = [ "https://1drv.ms/x/c/f6261b79731e452c/IQBpBPUwwVtQTYrz4LgAWE6ZAfTxxxhud5AzyDm4tVY5P-0?download=1",   # Dirty 1 - 24
         "https://1drv.ms/x/c/f6261b79731e452c/IQDTm4wgClDgRKutu40S_fAUASzQrkvm2Z--Qe2whPBX1xI?download=1"
         ]   # Dirty 25 - 107 

col = ['Date', 'Time', 'GPM_1', 'TOTAL_GAL_1', 'GPM_2','TOTAL_GAL_2', 'GPM_3', 'TOTAL_GAL_3', 'GPM_4', 'TOTAL_GAL_4', 'GPM_5','TOTAL_GAL_5', 'GPM_6', 'TOTAL_GAL_6', 'GPM_7', 'TOTAL_GAL_7', 'GPM_8','TOTAL_GAL_8', 'GPM_9', 'TOTAL_GAL_9', 'GPM_10', 'TOTAL_GAL_10', 'GPM_11','TOTAL_GAL_11', 'GPM_12', 'TOTAL_GAL_12', 'GPM_13', 'TOTAL_GAL_13','GPM_14', 'TOTAL_GAL_14', 'GPM_15', 'TOTAL_GAL_15', 'GPM_16','TOTAL_GAL_16', 'GPM_17', 'TOTAL_GAL_17', 'GPM_18', 'TOTAL_GAL_18','GPM_19', 'TOTAL_GAL_19', 'GPM_20', 'TOTAL_GAL_20', 'GPM_21','TOTAL_GAL_21', 'GPM_22', 'TOTAL_GAL_22', 'GPM_23', 'TOTAL_GAL_23','GPM_24', 'TOTAL_GAL_24', 'comments', 'datetime']
col2 = ['Date', 'Time','GPM_25', 'TOTAL_GAL_25', 'GPM_26','TOTAL_GAL_26','GPM_27', 'TOTAL_GAL_27', 'GPM_28','TOTAL_GAL_28', 'GPM_29', 'TOTAL_GAL_29', 'GPM_30','TOTAL_GAL_30', 'GPM_31', 'TOTAL_GAL_31', 'GPM_32', 'TOTAL_GAL_32', 'GPM_33','TOTAL_GAL_33', 'GPM_34', 'TOTAL_GAL_34', 'GPM_35', 'TOTAL_GAL_35', 'GPM_36','TOTAL_GAL_36', 'GPM_37', 'TOTAL_GAL_37', 'GPM_38', 'TOTAL_GAL_38', 'GPM_39','TOTAL_GAL_39', 'GPM_40', 'TOTAL_GAL_40', 'GPM_41', 'TOTAL_GAL_41','GPM_42', 'TOTAL_GAL_42', 'GPM_43', 'TOTAL_GAL_43', 'GPM_44','TOTAL_GAL_44', 'GPM_45', 'TOTAL_GAL_45', 'GPM_101', 'TOTAL_GAL_101','GPM_102', 'TOTAL_GAL_102', 'GPM_103', 'TOTAL_GAL_103', 'GPM_104','TOTAL_GAL_104', 'GPM_105', 'TOTAL_GAL_105', 'GPM_106', 'TOTAL_GAL_106','GPM_107', 'TOTAL_GAL_107', 'comments', 'datetime']

def get_data(url):
    response = requests.get(url)
    df = pd.read_excel(BytesIO(response.content), engine="openpyxl")
    return df

def shape_df(df):
    # cut the first 5 rows and first column 
    df = df.iloc[5:,1:].reset_index(drop=True)
    return df

def trim_frame(df, new_col):
    df = df.iloc[:, :len(new_col)] 
    df.columns = new_col
    return df

def clean_col(df):
    # strip all columns with "Unnamed" string
    for col in df.columns:
        if "Unnamed" in str(col):
            df = df.drop(columns=[str(col)], errors='ignore')
    return df

def fix_time(df):
    mask = df['Time'].apply(lambda x: isinstance(x, datetime))
    df.loc[mask, 'Time'] = df.loc[mask, 'Time'].apply(lambda x: x.time())
    return df

def make_datetime_col(df):
    df['datetime'] = df.apply(
        lambda row: datetime.combine(row['Date'].date(), row['Time']),
        axis=1
    )
    return df

def fix_space(df):
    df.columns = df.columns.str.replace('.', '_', regex=False)
    df.columns = df.columns.str.replace(' ', '_', regex=False)  # replace spaces
    return df

def drop_DT(df):
    # strip all columns with "Unnamed" string
    df = df.drop(columns=['Date', 'Time'], errors='ignore')
    return df


def merge_and_sort(df1, df2):
    df_merged = pd.merge(df1, df2, on='datetime', how='outer', suffixes=('_1', '_2'))
    df_merged.sort_values('datetime', inplace=True)
    df_merged.reset_index(drop=True, inplace=True)
    df_merged = df_merged.drop(columns=['comments_1', 'comments_2'], errors='ignore')
    return df_merged


def make_tidy(df):

    value_cols = [col for col in df.columns if 'GPM_' in col or 'TOTAL_GAL_' in col]

    df_long = df.melt(
        id_vars='datetime',
        value_vars=value_cols,
        var_name='variable',
        value_name='value'
    )

    # split column names
    df_long[['metric', 'pump']] = df_long['variable'].str.extract(r'(GPM|TOTAL_GAL)_(\d+)')
    df_long['pump'] = df_long['pump'].astype(int)

    # 🔑 FIX: force numeric
    df_long['value'] = pd.to_numeric(df_long['value'], errors='coerce')

    # 🔑 apply rule: GPM < 1 → 0
    df_long.loc[(df_long['metric'] == 'GPM') & (df_long['value'] < 1),'value'] = 0
    # pivot
    df_tidy = df_long.pivot_table(
        index=['datetime', 'pump'],
        columns='metric',
        values='value'
    ).reset_index()
    df_tidy.columns.name = None
    return df_tidy


def run_functions(url, list):
    df = get_data(url)
    df = shape_df(df)
    df = trim_frame(df, list)  # df2 check too 
    df = clean_col(df)
    df = fix_time(df)
    df = make_datetime_col(df)
    df = fix_space(df)
    df = drop_DT(df)
    print("The functions ran ....")
    return df


df1 = run_functions(live_urls[0], col)
df2 = run_functions(live_urls[1], col2)

df_combined = merge_and_sort(df1, df2)
df_tidy = make_tidy(df_combined)

df_tidy['datetime'] = pd.to_datetime(df_tidy['datetime'])



The functions ran ....
The functions ran ....


In [13]:
print(df_tidy.columns)


Index(['datetime', 'pump', 'GPM', 'TOTAL_GAL'], dtype='object')


In [26]:
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

def train_val_split(group, val_size=4):
    """
    Splits a single pump's data into train and validation sets.
    """
    train = group.iloc[:-val_size]
    val = group.iloc[-val_size:]
    return train, val

forecasts = {}
validation = {}

for pump_id, group in df_tidy.groupby('pump'):
    group = group.sort_values('datetime')
    train, val = train_val_split(group, val_size=4)
    
    # Save validation for comparison
    validation[pump_id] = val.set_index('datetime')['TOTAL_GAL']  # or TOTAL_GAL
    
    # Fit ARIMA on train
    from statsmodels.tsa.arima.model import ARIMA
    series = train.set_index('datetime')['TOTAL_GAL']  # target column
    
    try:
        model = ARIMA(series, order=(1,1,1))
        fitted = model.fit()
        
        # Forecast 4 steps (to match validation)
        forecast = fitted.forecast(steps=4)
        forecasts[pump_id] = forecast
        
    except Exception as e:
        print(f"Pump {pump_id} failed: {e}")

c:\Users\Maweeks\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Maweeks\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Maweeks\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Maweeks\AppData\Local\Programs\Python\Python314\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary

In [40]:
from pmdarima import auto_arima
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA

from pmdarima import auto_arima

forecasts = {}
validation = {}

for pump_id, group in df_tidy.groupby('pump'):
    group = group.sort_values('datetime')
    train = group.iloc[:-4]
    val = group.iloc[-4:]
    
    validation[pump_id] = val.set_index('datetime')['TOTAL_GAL']
    
    series = train.set_index('datetime')['TOTAL_GAL']
    series = series.asfreq('2h').ffill().bfill()  # critical fix
    
    try:
        model = auto_arima(series,
                           seasonal=False,
                           stepwise=True,
                           suppress_warnings=True)
        forecast = model.predict(n_periods=4)
        forecasts[pump_id] = pd.Series(forecast, index=val['datetime'])
    except Exception as e:
        print(f"Pump {pump_id} failed: {e}")

C:\Users\Maweeks\AppData\Local\Temp\ipykernel_2588\2125125337.py:19: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  series = series.asfreq('2H').ffill().bfill()  # critical fix
C:\Users\Maweeks\AppData\Local\Temp\ipykernel_2588\2125125337.py:19: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  series = series.asfreq('2H').ffill().bfill()  # critical fix
C:\Users\Maweeks\AppData\Local\Temp\ipykernel_2588\2125125337.py:19: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  series = series.asfreq('2H').ffill().bfill()  # critical fix
C:\Users\Maweeks\AppData\Local\Temp\ipykernel_2588\2125125337.py:19: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  series = series.asfreq('2H').ffill().bfill()  # critical fix
C:\Users\Maweeks\AppData\Local\Temp\ipykernel_2588\2125125337.py:19: FutureWarni

In [41]:
all_comparisons = []

for pump_id in validation.keys():
    val_series = validation[pump_id]
    forecast_series = forecasts[pump_id]

    comparison = pd.DataFrame({
        'Pump': pump_id,
        'Actual': val_series.values,
        'Forecasted': forecast_series.values
    }, index=val_series.index)

    comparison['Pct_Error'] = abs(comparison['Actual'] - comparison['Forecasted']) / comparison['Actual'] * 100
    all_comparisons.append(comparison)

comparison_df = pd.concat(all_comparisons).reset_index().rename(columns={'index':'datetime'})
print(comparison_df)

               datetime  Pump     Actual    Forecasted  Pct_Error
0   2026-04-06 19:00:00     1  5817897.0  5.817416e+06   0.008259
1   2026-04-06 21:00:00     1  5821355.0  5.820061e+06   0.022236
2   2026-04-06 23:00:00     1  5824983.0  5.821988e+06   0.051421
3   2026-04-07 01:00:00     1  5828440.0  5.826307e+06   0.036590
4   2026-04-06 19:00:00     2  4933159.0  4.933265e+06   0.002157
..                  ...   ...        ...           ...        ...
203 2026-04-07 01:00:00   106    53260.0  5.316138e+04   0.185168
204 2026-04-06 19:00:00   107   578379.0  5.777783e+05   0.103862
205 2026-04-06 21:00:00   107   579618.0  5.785344e+05   0.186952
206 2026-04-06 23:00:00   107   581003.0  5.794640e+05   0.264891
207 2026-04-07 01:00:00   107   582198.0  5.804948e+05   0.292546

[208 rows x 5 columns]


In [44]:
comparison_df.to_csv('modeled')

In [ ]:
# # Loop over each pump and show validation vs forecast
# for pump_id in validation.keys():
#     val_series = validation[pump_id]
#     forecast_series = forecasts[pump_id]
    
#     print(f"\nPump {pump_id} - Last 4 Values vs Forecast")
#     comparison = pd.DataFrame({
#         'Validation': val_series.values,
#         'Forecast': forecast_series.values
#     }, index=val_series.index)
    
#     print(comparison)


Pump 1 - Last 4 Values vs Forecast
                     Validation      Forecast
datetime                                     
2026-04-06 19:00:00   5817897.0  5.818049e+06
2026-04-06 21:00:00   5821355.0  5.821579e+06
2026-04-06 23:00:00   5824983.0  5.825110e+06
2026-04-07 01:00:00   5828440.0  5.828640e+06

Pump 2 - Last 4 Values vs Forecast
                     Validation      Forecast
datetime                                     
2026-04-06 19:00:00   4933159.0  4.933267e+06
2026-04-06 21:00:00   4936084.0  4.936203e+06
2026-04-06 23:00:00   4939190.0  4.939140e+06
2026-04-07 01:00:00   4942127.0  4.942077e+06

Pump 3 - Last 4 Values vs Forecast
                     Validation      Forecast
datetime                                     
2026-04-06 19:00:00   3725175.0  3.725246e+06
2026-04-06 21:00:00   3727290.0  3.727383e+06
2026-04-06 23:00:00   3729551.0  3.729521e+06
2026-04-07 01:00:00   3731669.0  3.731659e+06

Pump 4 - Last 4 Values vs Forecast
                     Validat

In [ ]:
# all_comparisons = []

# for pump_id in validation.keys():
#     val_series = validation[pump_id]
#     forecast_series = forecasts[pump_id]

#     comparison = pd.DataFrame({
#         'Pump': pump_id,
#         'Validation': val_series.values,
#         'Forecast': forecast_series.values
#     }, index=val_series.index)

#     # Add simple accuracy metrics
#     comparison['Error'] = comparison['Validation'] - comparison['Forecast']       # raw error
#     comparison['Abs_Error'] = comparison['Error'].abs()                            # absolute error
#     comparison['Pct_Error'] = (comparison['Abs_Error'] / comparison['Validation']) * 100  # percent error

#     all_comparisons.append(comparison)

# # Combine all pumps into one DataFrame
# comparison_df = pd.concat(all_comparisons)
# comparison_df.index.name = 'datetime'
# comparison_df.reset_index(inplace=True)
# print(comparison_df)

               datetime  Pump  Validation      Forecast       Error  \
0   2026-04-06 19:00:00     1   5817897.0  5.818049e+06 -151.751163   
1   2026-04-06 21:00:00     1   5821355.0  5.821579e+06 -224.381922   
2   2026-04-06 23:00:00     1   5824983.0  5.825110e+06 -126.892282   
3   2026-04-07 01:00:00     1   5828440.0  5.828640e+06 -200.282246   
4   2026-04-06 19:00:00     2   4933159.0  4.933267e+06 -107.662539   
..                  ...   ...         ...           ...         ...   
203 2026-04-07 01:00:00   106     53260.0  5.326086e+04   -0.856617   
204 2026-04-06 19:00:00   107    578379.0  5.784504e+05  -71.377059   
205 2026-04-06 21:00:00   107    579618.0  5.797257e+05 -107.703714   
206 2026-04-06 23:00:00   107    581003.0  5.810010e+05    2.020032   
207 2026-04-07 01:00:00   107    582198.0  5.822762e+05  -78.205823   

      Abs_Error  Pct_Error  
0    151.751163   0.002608  
1    224.381922   0.003854  
2    126.892282   0.002178  
3    200.282246   0.003436  
4 

In [43]:
clean_df = pd.DataFrame({
    'Pump': comparison_df['Pump'],
    # 'Actual': comparison_df['Validation'],
    'Forecasted': comparison_df['Forecast'],
    'Pct_Error': comparison_df['Pct_Error'] * 100  # convert fraction to %
})

# Optional: reset index
clean_df.reset_index(drop=True, inplace=True)

# Preview
print(clean_df.head())

KeyError: 'Forecast'

In [32]:
clean_df.to_csv('clean')